# PDFs de dry spells: efecto de la calibración de umbral

## Objetivo
En **3 cuencas** (Loa, Maule, Biobío), comparar en un mismo gráfico la PDF de duraciones de dry spells bajo **cuatro configuraciones de umbral** (solo histórico 1980–2014):

| Curva | Producto | Umbral día seco |
|-------|----------|-----------------|
| **CR2MET 1 mm** | CR2MET | pr < 1 mm/día |
| **ALADIN 1 mm** | ALADIN hist | pr < 1 mm/día (sin calibrar) |
| **ALADIN τ* global** | ALADIN hist | pr < 5,285 mm/día (Pregunta 6) |
| **ALADIN τ* local** | ALADIN hist | pr < τ*(x,y) por píxel (Pregunta 6/7, criterio iii) |

Complementa `pregunta9_pdf_dryspells_cuencas.ipynb`, donde cada criterio va en figuras separadas (con futuro SSP5-8.5). Aquí el foco es **visualizar el efecto de calibrar el umbral en ALADIN** frente a la referencia CR2MET.

## Metodología
- Pool regional por cuenca (todos los píxeles × todo el periodo).
- NaN no cuentan como secos; spells censurados al inicio/fin excluidos.
- Histograma: bins log en X; altura = conteos / ancho de bin ⇒ **∫ f(t) dt = n_spells**.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.io.shapereader as shpreader
from IPython.display import display
from shapely.geometry import Point
from shapely.ops import unary_union
from shapely.prepared import prep

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

HIST_START = '1980-01-01'
HIST_END = '2014-12-31'
TAU_CR2MET_REF = 1.0
TAU_ALADIN_DOMINIO = 5.285
BISECTION_TAU_MAX = 15.0
BISECTION_TOL = 1e-4
PDF_BINS = 25
MIN_DURATION = 1
EXCLUDE_BOUNDARY_SPELLS = True
OUT_DIR = Path('_pregunta9_calibracion_outputs')
OUT_DIR.mkdir(exist_ok=True)

BASIN_SPECS = {
    'Loa':    {'bounds': (-69.8, -68.2, -24.0, -21.5)},
    'Maule':  {'bounds': (-72.8, -71.0, -36.5, -34.8)},
    'Biobio': {'bounds': (-73.8, -71.5, -38.5, -36.5)},
}

SERIES_CONFIG = [
    ('CR2MET 1 mm',        'cr2met', 1.0,    {'color': 'black',     'ls': '-',  'lw': 2.4}),
    ('ALADIN 1 mm',        'aladin', 'same', {'color': 'crimson',   'ls': '--', 'lw': 2.0}),
    ('ALADIN tau* global', 'aladin', 'global', {'color': 'steelblue', 'ls': '-',  'lw': 2.0}),
    ('ALADIN tau* local',  'aladin', 'local',  {'color': 'forestgreen', 'ls': '-.', 'lw': 2.0}),
]


def load_chile_geometry():
    reader = shpreader.Reader(
        shpreader.natural_earth(resolution='10m', category='cultural', name='admin_0_countries')
    )
    geoms = [r.geometry for r in reader.records()
             if r.attributes.get('NAME') == 'Chile' or r.attributes.get('ADMIN') == 'Chile']
    return unary_union(geoms)


def build_chile_mask_on_aladin_grid(lat2d, lon2d, geometry):
    prepared = prep(geometry)
    flat = np.fromiter(
        (prepared.contains(Point(float(x), float(y))) or geometry.touches(Point(float(x), float(y)))
         for y, x in zip(lat2d.ravel(), lon2d.ravel())),
        dtype=bool, count=lat2d.size,
    )
    return flat.reshape(lat2d.shape)


def build_basin_mask(lat2d, lon2d, chile_mask_bool, bounds):
    lon_min, lon_max, lat_min, lat_max = bounds
    return (
        (lon2d >= lon_min) & (lon2d <= lon_max) &
        (lat2d >= lat_min) & (lat2d <= lat_max) & chile_mask_bool
    )


def open_aladin_hist(start, end):
    files = sorted(Path('pr1').glob('pr_CHP12_*_historical_*.nc'))
    ds = xr.open_mfdataset([str(p) for p in files], use_cftime=True, chunks={'time': 365})
    return ds['pr'].sel(time=slice(start, end)) * 86400.0


def open_cr2met_hist(start, end):
    ds = xr.open_mfdataset('./pr/CR2MET_pr_v2.5_day_*.nc', chunks={'time': 365})
    return ds['pr'].sel(time=slice(start, end))


print('1/3: Cargando datos historicos...')
pr_ala_hist = open_aladin_hist(HIST_START, HIST_END)
pr_cr2_hist = open_cr2met_hist(HIST_START, HIST_END).interp(
    lat=pr_ala_hist['lat'], lon=pr_ala_hist['lon'], method='linear'
)

chile_geom = load_chile_geometry()
lat2d = pr_ala_hist['lat'].values
lon2d = pr_ala_hist['lon'].values
chile_mask_bool = build_chile_mask_on_aladin_grid(lat2d, lon2d, chile_geom)
chile_mask = xr.DataArray(
    chile_mask_bool,
    coords={'y': pr_ala_hist['y'], 'x': pr_ala_hist['x'],
            'lat': pr_ala_hist['lat'], 'lon': pr_ala_hist['lon']},
    dims=['y', 'x'], name='chile_mask',
)

basin_masks = {}
for name, spec in BASIN_SPECS.items():
    bm = build_basin_mask(lat2d, lon2d, chile_mask_bool, spec['bounds'])
    basin_masks[name] = xr.DataArray(bm, coords=chile_mask.coords, dims=['y', 'x'])
    print(f'  {name}: {int(bm.sum())} pixeles')
print(f'Periodo: {HIST_START[:4]}-{HIST_END[:4]}')

In [ ]:
# =====================================================================
# EXTRACCION DE DRY SPELLS + POOL REGIONAL
# =====================================================================
def wet_fraction_1d(pr_series, threshold):
    x = np.asarray(pr_series, dtype=float)
    x = x[np.isfinite(x)]
    return float((x >= threshold).mean()) if x.size else np.nan


def find_threshold_for_target_1d(pr_series, target_fraction, tau_max=BISECTION_TAU_MAX, tol=BISECTION_TOL):
    if not np.isfinite(target_fraction):
        return np.nan
    f0, fmax = wet_fraction_1d(pr_series, 0.0), wet_fraction_1d(pr_series, tau_max)
    if target_fraction > f0 + tol or target_fraction < fmax - tol:
        return np.nan
    lo, hi = 0.0, tau_max
    for _ in range(80):
        mid = 0.5 * (lo + hi)
        fmid = wet_fraction_1d(pr_series, mid)
        if abs(fmid - target_fraction) <= tol:
            return mid
        lo, hi = (mid, hi) if fmid > target_fraction else (lo, mid)
    return 0.5 * (lo + hi)


def pixelwise_local_aladin_threshold(pr_cr2, pr_aladin, tau_cr2, mask_da):
    f_target = (pr_cr2 >= tau_cr2).mean(dim='time').where(mask_da)
    stacked = xr.Dataset({
        'pr_ala': pr_aladin.stack(cell=('y', 'x')),
        'f_target': f_target.stack(cell=('y', 'x')),
    }).compute()
    n_cells = stacked.sizes['cell']
    tau_star = np.full(n_cells, np.nan, dtype=np.float32)
    for idx in range(n_cells):
        ft = float(stacked['f_target'].values[idx])
        if np.isfinite(ft):
            tau_star[idx] = find_threshold_for_target_1d(stacked['pr_ala'].values[:, idx], ft)
    tau_da = xr.DataArray(tau_star, coords={'cell': stacked['cell']}, dims=['cell']).unstack('cell')
    return tau_da.assign_coords(lat=mask_da['lat'], lon=mask_da['lon'])


def time_to_year(t):
    if hasattr(t, 'year'):
        return int(t.year)
    return int(pd.Timestamp(t).year)


def dry_bool_1d(col):
    col = np.asarray(col, dtype=float)
    return np.isfinite(col) & (col > 0.5)


def run_lengths_1d(dry_bool):
    dry_bool = np.asarray(dry_bool, dtype=bool)
    if not np.any(dry_bool):
        return np.array([], dtype=np.int32), np.array([], dtype=np.int32), np.array([], dtype=np.int32)
    padded = np.r_[False, dry_bool, False]
    dx = np.diff(padded.astype(np.int8))
    starts, ends = np.where(dx == 1)[0], np.where(dx == -1)[0]
    return starts, ends, (ends - starts).astype(np.int32)


def extract_basin_spell_records(pr, dry_threshold, basin_mask, exclude_boundary=EXCLUDE_BOUNDARY_SPELLS):
    is_dry = (pr < dry_threshold).where(basin_mask)
    stacked = is_dry.stack(cell=('y', 'x')).transpose('time', 'cell').compute()
    times = stacked['time'].values
    vals = stacked.values
    mask_flat = basin_mask.stack(cell=('y', 'x')).values
    n_time = vals.shape[0]
    records = []
    for idx in range(vals.shape[1]):
        if not mask_flat[idx]:
            continue
        dry = dry_bool_1d(vals[:, idx])
        if not np.any(dry):
            continue
        starts, ends, durations = run_lengths_1d(dry)
        for s, e, duration in zip(starts, ends, durations):
            if exclude_boundary and (s == 0 or e == n_time):
                continue
            if duration > 0:
                records.append({'duration': int(duration)})
    return pd.DataFrame(records)


def spell_durations(spell_df):
    if spell_df.empty:
        return np.array([], dtype=np.int32)
    return spell_df['duration'].to_numpy(dtype=np.int32)


def histogram_density_nspells(durations, n_bins=PDF_BINS, min_duration=MIN_DURATION):
    d = np.asarray(durations, dtype=float)
    d = d[np.isfinite(d) & (d >= min_duration)]
    if d.size < 50:
        return None, None, 0
    lo = max(float(np.min(d)), min_duration)
    hi = float(np.max(d))
    if hi <= lo:
        return None, None, 0
    bins = np.logspace(np.log10(lo), np.log10(hi), n_bins + 1)
    counts, edges = np.histogram(d, bins=bins, density=False)
    widths = np.diff(edges)
    centers = np.sqrt(edges[:-1] * edges[1:])
    heights = counts / widths
    return centers, heights, int(d.size)


def summarize_durations(durations):
    d = np.asarray(durations, dtype=float)
    d = d[np.isfinite(d) & (d >= 1)]
    if d.size == 0:
        return {}
    return {
        'n_spells': int(d.size),
        'mean_dias': float(np.mean(d)),
        'median_dias': float(np.median(d)),
        'p90_dias': float(np.percentile(d, 90)),
        'p99_dias': float(np.percentile(d, 99)),
        'max_dias': float(np.max(d)),
    }


THRESH_ALA = {
    'same': TAU_CR2MET_REF,
    'global': TAU_ALADIN_DOMINIO,
}

print('2/3: tau* local + extraccion por cuenca (4 curvas)...')
tau_local_map = pixelwise_local_aladin_threshold(
    pr_cr2_hist.where(chile_mask), pr_ala_hist.where(chile_mask), TAU_CR2MET_REF, chile_mask,
)
THRESH_ALA['local'] = tau_local_map

basin_durations = {}
summary_rows = []

for basin_name, basin_mask in basin_masks.items():
    durs = {}
    for label, product, thr_key, _ in SERIES_CONFIG:
        if product == 'cr2met':
            thr = TAU_CR2MET_REF
            pr = pr_cr2_hist
        else:
            thr = THRESH_ALA[thr_key]
            pr = pr_ala_hist
        arr = spell_durations(extract_basin_spell_records(pr, thr, basin_mask))
        durs[label] = arr
        s = summarize_durations(arr)
        if s:
            summary_rows.append({'cuenca': basin_name, 'serie': label, **s})
    basin_durations[basin_name] = durs

summary_df = pd.DataFrame(summary_rows)
print('Resumen por cuenca y configuracion de umbral:')
display(summary_df.round(1))
summary_df.round(3).to_csv(OUT_DIR / 'resumen_calibracion_umbral.csv', index=False)
print(f'Guardado: {OUT_DIR / "resumen_calibracion_umbral.csv"}')

In [ ]:
# =====================================================================
# GRAFICOS PDF — comparacion de umbrales en un solo panel
# =====================================================================
def plot_calibration_pdf(basin_name, durs_dict, log_y=False, save_name=None):
    fig, ax = plt.subplots(figsize=(10, 6))
    for label, _, _, style in SERIES_CONFIG:
        centers, heights, n_spells = histogram_density_nspells(durs_dict[label])
        if centers is None:
            continue
        yvals = heights.copy()
        if log_y:
            yvals = np.where(yvals > 0, yvals, np.nan)
        ax.plot(centers, yvals, label=f'{label} (n={n_spells:,})', **style)

    ax.set_xscale('log')
    if log_y:
        ax.set_yscale('log')
    ax.set_xlabel('Duracion de dry spell (dias)')
    ax.set_ylabel('f(t)  [integral = n_spells]' + (' [log]' if log_y else ''))
    ax.set_title(
        f'Efecto calibracion de umbral — Cuenca {basin_name}\n'
        f'CR2MET 1 mm vs ALADIN 1 mm / tau* global (5.285) / tau* local | {HIST_START[:4]}-{HIST_END[:4]}',
        fontweight='bold',
    )
    ax.legend(fontsize=9)
    plt.tight_layout()
    if save_name:
        fig.savefig(OUT_DIR / save_name, dpi=150, bbox_inches='tight')
    plt.show()


print('3/3a: PDFs eje Y lineal (3 cuencas)...')
for basin_name in BASIN_SPECS:
    plot_calibration_pdf(
        basin_name, basin_durations[basin_name], log_y=False,
        save_name=f'pdf_calibracion_{basin_name}_lineal.png',
    )

print('3/3b: PDFs eje Y logaritmico (3 cuencas)...')
for basin_name in BASIN_SPECS:
    plot_calibration_pdf(
        basin_name, basin_durations[basin_name], log_y=True,
        save_name=f'pdf_calibracion_{basin_name}_logy.png',
    )

print(f'Figuras en: {OUT_DIR.resolve()}')

## Lectura esperada

- **ALADIN 1 mm** (rojo): rachas mas cortas que CR2MET → sesgo por umbral no calibrado.
- **ALADIN tau* global** (azul): mas cerca de CR2MET que con 1 mm.
- **ALADIN tau* local** (verde): ajuste pixel a pixel; suele acercarse mas a CR2MET en cuencas centrales/sur.

### Equivalencia con `pregunta9_pdf_dryspells_cuencas.ipynb`

| Esta notebook | Pregunta 9 |
|---------------|------------|
| CR2MET 1 mm | CR2MET en criterios i / ii / iii |
| ALADIN 1 mm | ALADIN en criterio **ii** (mismo umbral) |
| ALADIN tau* global | ALADIN en criterio **i** |
| ALADIN tau* local | ALADIN en criterio **iii** |